# 🏴‍☠️ AN2DL Challenge 1: Pirate Pain Dataset

## 🧠 Notebook 03: Model Definition

In [ ]:
import json
from itertools import product
from time import sleep

import GPUtil
import numpy as np
import pandas as pd
import psutil
import torch
import torch.nn as nn
from sklearn.metrics import f1_score, recall_score, confusion_matrix
from sklearn.model_selection import StratifiedKFold
from sklearn.utils.class_weight import compute_class_weight
from torch.nn.functional import gelu
from torch.utils.data import WeightedRandomSampler, DataLoader, Dataset

import internal.telegram_utils as tg
from internal.utils import PersistenceManager

### Model components

The model consists of several components:
1. **TCN Building Blocks**: Dilated 1D convolutional blocks with residual connections.
2. **Attention Pooling**: An attention mechanism to pool features over the time dimension.
3. **Static MLP**: A feedforward network to process static features.
4. **PainTCNBiLSTMAttn**: The full model that integrates all components.

Additionally, we set up the loss function, optimizer, and learning rate scheduler for training.

In [ ]:
# --- TCN building blocks ---

class Conv1dBlock(nn.Module):
    """
    Dilated 1D conv + GroupNorm(=LayerNorm-like) + GELU + Dropout, with residual.
    Input/Output: (B, C, T)
    """
    def __init__(self, channels: int, kernel_size: int = 5, dilation: int = 1, dropout: float = 0.3):
        super().__init__()
        pad = (kernel_size - 1) * dilation // 2  # 'same' padding
        self.conv = nn.Conv1d(channels, channels, kernel_size, padding=pad, dilation=dilation)
        self.norm = nn.GroupNorm(1, channels)  # LayerNorm-like per-channel across (B,T)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):         # x: (B,C,T)
        residual = x
        x = self.conv(x)
        x = self.norm(x)
        x = gelu(x)
        x = self.drop(x)
        return x + residual       # residual

class TCN(nn.Module):
    """Stack of dilated residual Convolution 1d blocks."""
    def __init__(self, channels: int, dilations=(1,2,4,8), kernel_size: int = 5, dropout: float = 0.3):
        super().__init__()
        self.blocks = nn.ModuleList([
            Conv1dBlock(channels, kernel_size=kernel_size, dilation=d, dropout=dropout) for d in dilations
        ])
    def forward(self, x):         # (B,C,T)
        for b in self.blocks:
            x = b(x)
        return x

# --- Attention pooling over time ---

class AttnPool(nn.Module):
    """
    Additive attention over time.
    Input: (B, T, C)
    Output: (B, C)
    """
    def __init__(self, in_dim: int, hidden: int = 128):
        super().__init__()
        self.score = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.Tanh(),
            nn.Linear(hidden, 1)
        )
    def forward(self, x):         # x: (B,T,C)
        a = self.score(x).squeeze(-1)         # (B,T)
        a = torch.softmax(a, dim=1)           # (B,T)
        return torch.sum(x * a.unsqueeze(-1), dim=1)  # (B,C)

# --- Static MLP ---

class StaticMLP(nn.Module):
    def __init__(self, in_dim=7, hidden=64, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.RReLU(0.1, 0.3), nn.Dropout(dropout),
            nn.Linear(hidden, hidden), nn.RReLU(0.1, 0.3), nn.Dropout(dropout)
        )
    def forward(self, x):         # (B,7)
        return self.net(x)        # (B,64)

# --- Full model ---

class PainTCNBiLSTMAttn(nn.Module):
    """
    Inputs:
      x_num:  (B, T=160, D_num)    # engineered numeric dynamics: 90 (30 raw + 30 delta + 30 rollstd)
      x_surv: list of 4 (B, T)     # each int in {0,1,2} for pain_survey_1..4
      x_sta:  (B, 7)               # static features (scaled joint_30 + counts + binaries)
      x_summ: (B, F_summ)          # summary features
    """
    def __init__(self,
                 d_num: int = 90,
                 d_emb: int = 2,
                 d_sta: int = 7,
                 d_summ: int = 0,
                 tcn_channels: int = 128,
                 lstm_hidden: int = 128,
                 num_classes: int = 3,
                 dropout: float = 0.3,
                 p_drop_summ=0.5):
        super().__init__()

        # 4 survey embeddings (3 categories each)
        self.survey_embs = nn.ModuleList([nn.Embedding(3, d_emb) for _ in range(4)])

        # Project timestep features to TCN channels
        d_in = d_num + 4 * d_emb
        self.proj = nn.Linear(d_in, tcn_channels)

        # TCN front
        self.tcn = TCN(channels=tcn_channels, dilations=(1,2,4,8), kernel_size=5, dropout=dropout)

        # BiLSTM over (B,T,C)
        self.lstm = nn.LSTM(input_size=tcn_channels, hidden_size=lstm_hidden,
                            num_layers=1, batch_first=True, bidirectional=True, dropout=0.0)

        # Attention pooling over time
        self.attn = AttnPool(in_dim=2*lstm_hidden, hidden=128)

        # Static branch
        self.static_mlp = StaticMLP(in_dim=d_sta, hidden=64, dropout=dropout)

        # Summary features branch
        self.summ_out = summ_out = 16 # the number of each summary feature output dim
        self.summ_mlp = nn.Sequential(
            nn.LayerNorm(d_summ),
            nn.Linear(d_summ, 128), nn.RReLU(0.1, 0.3), nn.Dropout(dropout),
            nn.Linear(128, summ_out), nn.RReLU(0.1, 0.3)
        )

        self.p_drop_summ = p_drop_summ

        # Fusion gate
        self.gate_fc = nn.Linear(64 + summ_out, 2*lstm_hidden) # from static + summary to gate for dynamic

        # Fusion head
        fused_in = (2*lstm_hidden) + 64 + summ_out  # dyn + sta + summ
        self.head = nn.Sequential(
            nn.Linear(fused_in, 128),
            nn.RReLU(0.1, 0.3), nn.Dropout(dropout),
            nn.Linear(128, num_classes)
        )

    def forward(self, _x_num, x_surv_list, _x_sta, _x_summ):
        """
        x_num: (B,T,D_num)
        x_surv_list: [s1,s2,s3,s4] each (B,T) long
        x_sta: (B,7)
        """
        B, T, _ = _x_num.shape
        # Embeddings for the 4 surveys
        embs = [emb(surv) for emb, surv in zip(self.survey_embs, x_surv_list)]  # 4 × (B,T,d_emb)
        x = torch.cat([_x_num, *embs], dim=-1)                                   # (B,T, d_num+4*d_emb)

        # Project to TCN channels
        x = self.proj(x)                       # (B,T,C)
        # TCN expects (B,C,T)
        x = x.transpose(1, 2)                  # (B,C,T)
        x = self.tcn(x)                        # (B,C,T)
        x = x.transpose(1, 2)                  # (B,T,C)

        # BiLSTM
        x, _ = self.lstm(x)                    # (B,T,2*hidden)

        # Attention pooling
        dyn_vec = self.attn(x)                 # (B,2*hidden)
        # Static branch
        sta_vec = self.static_mlp(_x_sta)       # (B,64)
        # Summary features branch
        summ_vec = self.summ_mlp(_x_summ)  # (B,64)

        # This forces the dynamic+static paths to stay strong and prevents shortcutting
        if self.training and torch.rand(1, device=summ_vec.device) < self.p_drop_summ:
            summ_vec = torch.zeros_like(summ_vec)  # drop the entire branch this step

        # --- Gated fusion ---
        # Concatenate context (static + summary) and generate a gate for the dynamic vector
        context = torch.cat([sta_vec, summ_vec], dim=-1)    # (B,128)
        gate = torch.sigmoid(self.gate_fc(context))         # (B, 2H)
        dyn_vec *= gate                                     # modulate dynamic encoding

        # Final fusion
        z = torch.cat([dyn_vec, sta_vec, summ_vec], dim=-1)
        _logits = self.head(z)                       # (B,3)

        return _logits


In [ ]:
class LabelSmoothingCE(nn.Module):
    def __init__(self, smoothing=0.05, weight=None):
        super().__init__()
        self.smoothing = smoothing
        self.register_buffer('weight', weight if weight is not None else None)

    def forward(self, _logits, target):
        n_class = _logits.size(-1)
        log_prob = torch.log_softmax(_logits, dim=-1)
        with torch.no_grad():
            true_dist = torch.zeros_like(log_prob)
            true_dist.fill_(self.smoothing / (n_class - 1))
            true_dist.scatter_(1, target.unsqueeze(1), 1.0 - self.smoothing)
        # class weights applied as expected value over classes
        if self.weight is not None:
            w = self.weight.unsqueeze(0)  # (1,C)
            _loss = (-true_dist * log_prob * w).sum(dim=-1)
        else:
            _loss = (-true_dist * log_prob).sum(dim=-1)
        return _loss.mean()

class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, reduction='mean'):
        super().__init__()
        self.gamma = gamma
        self.reduction = reduction
        self.register_buffer('alpha', alpha if alpha is not None else None)

    def forward(self, _logits, target):
        logpt = torch.log_softmax(_logits, dim=-1)           # (B,C)
        pt    = torch.exp(logpt)                             # (B,C)
        # gather log-prob and prob of the true class
        logpt_t = logpt.gather(1, target.unsqueeze(1)).squeeze(1)  # (B,)
        pt_t    = pt.gather(1, target.unsqueeze(1)).squeeze(1)     # (B,)
        _loss = - (1 - pt_t) ** self.gamma * logpt_t                # (B,)
        if self.alpha is not None:
            alpha_t = self.alpha[target]                            # (B,)
            _loss = alpha_t * _loss
        if self.reduction == 'mean':
            return _loss.mean()
        elif self.reduction == 'sum':
            return _loss.sum()
        else:
            return _loss

def normalize_class_weights(w):
    w = w / w.sum()
    return w

def make_weighted_sampler(y_train_fold, _class_weights):
    # map each sample to its class weight
    weights = _class_weights.cpu().numpy()
    sample_w = np.take(weights, y_train_fold)  # (N,)
    return WeightedRandomSampler(weights=sample_w, num_samples=len(sample_w), replacement=True)


In [ ]:
class PainDataset(Dataset):
    def __init__(self, _x_dyn_num, _x_surv, _x_sta, _x_summ, _y=None):
        self.X_dyn_num = torch.from_numpy(_x_dyn_num).float()
        self.X_sta     = torch.from_numpy(_x_sta).float()
        self.X_surv    = [torch.from_numpy(s).long() for s in _x_surv] if _x_surv is not None else None
        self.x_summ    = torch.from_numpy(_x_summ).float()
        self.y         = torch.from_numpy(_y).long() if _y is not None else None

    def __len__(self):
        return self.X_dyn_num.shape[0]

    def __getitem__(self, idx):
        item = {
            'x_num':  self.X_dyn_num[idx],              # (160, 90)
            'x_surv': [s[idx] for s in self.X_surv],    # list of 4 (160,)
            'x_sta':  self.X_sta[idx],                  # (7,)
            'x_summ': self.x_summ[idx],                # (varies)
        }
        if self.y is not None:
            item['y'] = self.y[idx]                     # ()
        return item


In [ ]:
# Robust loading: try v2, otherwise fall back to the older loader.
# Remove the variable annotation to avoid accidental instantiation issues.
data = PersistenceManager.load_arrays_v2()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

train_dataset = PainDataset(
    data.X_dyn_num_train,
    data.X_surv_train,
    data.X_sta_train,
    data.X_dyn_summ_train,
    data.y_train
)
val_dataset   = PainDataset(
    data.X_dyn_num_val,
    data.X_surv_val,
    data.X_sta_val,
    data.X_dyn_summ_val,
    data.y_val
)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,          # randomize samples each epoch
    num_workers=4,
    pin_memory=True
)
val_loader = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

# Example batch shapes:
# x_num:  (B,160,90)      -> float32
# x_surv: [ (B,160), ...] -> int64
# x_sta:  (B,7)           -> float32
# y:      (B,)            -> int64 in {0,1,2}

def make_class_weights(y_train_fold, _device):
    classes = np.array([0,1,2])
    w = compute_class_weight(class_weight='balanced', classes=classes, y=y_train_fold)
    w[2] *= 1.25   # try 1.2-1.5; watch that precision doesn’t collapse
    w = torch.tensor(w, dtype=torch.float32, device=_device)
    return w

# Weighted CE
# class_weights = make_class_weights(data.y_train, device)
# criterion = nn.CrossEntropyLoss(weight=class_weights)
# TODO: Sometimes stabilizes with imbalance, use instead of plain CE:
# criterion = LabelSmoothingCE(smoothing=0.05, weight=class_weights)
# TODO: Use if high_pain recall < 0.55 after a few epochs:
#       don't combine Focal + WeightedRandomSampler at the same time initially. Try Focal or Sampler.
# def effective_num_weights(_y, num_classes=3, beta=0.999):
#     cnt = np.bincount(_y, minlength=num_classes).astype(np.float32)
#     eff = 1.0 - np.power(beta, cnt)
#     w = (1.0 - beta) / (eff + 1e-8)
#     w = w / w.sum()  # normalize to sum=1 for α
#     return w
# alpha_np = effective_num_weights(data.y_train, 3, beta=0.999)
# alpha = torch.tensor(alpha_np, dtype=torch.float32, device=device)
# alpha = normalize_class_weights(class_weights)     # tensor on device
# criterion = FocalLoss(alpha=alpha, gamma=2.0)
# TODO: Turn this on only if CE (or LS-CE) still under-recalls high_pain.
#       If you use the sampler, keep the loss as plain CE or LabelSmoothingCE (not focal), to avoid over-focusing instability.
# sampler = make_weighted_sampler(data.y_train, class_weights)
# train_loader = DataLoader(train_dataset, batch_size=64, sampler=sampler, num_workers=4, pin_memory=True)
# sample_w = torch.tensor(class_weights, dtype=torch.float32)[torch.from_numpy(data.y_train)]
# sampler = torch.utils.data.WeightedRandomSampler(weights=sample_w.double(),
#                                                  num_samples=len(sample_w), replacement=True)
# train_loader = DataLoader(train_dataset, batch_size=64, sampler=sampler, num_workers=4, pin_memory=True)

# Optimizer / schedule
# optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
# scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5, min_lr=2e-5)


In [ ]:
@torch.no_grad()
def evaluate_metrics(_model, _loader, _device):
    _model.eval()
    y_true, y_pred = [], []
    for _batch in _loader:
        _x_num  = _batch['x_num'].to(_device)         # (B,160,90)
        _x_sta  = _batch['x_sta'].to(_device)         # (B,7)
        _x_surv = [s.to(_device) for s in _batch['x_surv']]  # list of 4 (B,160)
        _x_summ = _batch['x_summ'].to(_device)          # (B,F_summ)
        _y      = _batch['y'].to(_device)             # (B,)
        _logits = _model(_x_num, _x_surv, _x_sta, _x_summ)       # (B,3)
        y_hat  = _logits.argmax(dim=1)
        y_true.append(_y.cpu().numpy())
        y_pred.append(y_hat.cpu().numpy())
    y_true = np.concatenate(y_true)
    y_pred = np.concatenate(y_pred)

    _macro_f1  = f1_score(y_true, y_pred, average='macro', labels=[0,1,2])
    _rec_pc    = recall_score(y_true, y_pred, average=None, labels=[0,1,2])  # per-class recall
    _cm        = confusion_matrix(y_true, y_pred, labels=[0,1,2])
    return _macro_f1, _rec_pc, _cm

@torch.no_grad()
def infer_logits(_model, _x_num, _x_surv, _x_sta, _x_summ, batch_size=64):
    _model.eval()
    ds = PainDataset(_x_num, _x_surv, _x_sta, _x_summ)
    dl = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)
    preds = []
    for b in dl:
        x_num  = b['x_num'].to(device)
        x_sta  = b['x_sta'].to(device)
        x_surv = [s.to(device) for s in b['x_surv']]
        x_summ = b['x_summ'].to(device)
        logits = _model(x_num, x_surv, x_sta, x_summ)  # (B,3)
        preds.append(logits.detach().cpu().numpy())
    return np.concatenate(preds, axis=0)  # (N,3)

@torch.no_grad()
def infer_logits_only(_model, _loader, _device):
    _model.eval()
    outs = []
    for _batch in _loader:
        x_num  = _batch['x_num'].to(_device)
        x_sta  = _batch['x_sta'].to(_device)
        x_surv = [s.to(_device) for s in _batch['x_surv']]
        x_summ = _batch['x_summ'].to(_device)
        logits = _model(x_num, x_surv, x_sta, x_summ)  # (B,3)
        outs.append(logits.detach().cpu().numpy())
    return np.concatenate(outs, axis=0)

def search_logit_bias(_model, _val_loader, _y_val, _device):
    logits_val = infer_logits_only(_model, _val_loader, _device)  # (N,3)
    grid = np.linspace(-0.6, 0.6, 13)
    best_f1, best_b = -1., np.zeros(3, dtype=np.float32)
    for b in product(grid, repeat=3):
        b = np.array(b, dtype=np.float32)
        pred = (logits_val + b[None, :]).argmax(1)
        _f1 = f1_score(_y_val, pred, average='macro', labels=[0,1,2])
        if _f1 > best_f1:
            best_f1, best_b = _f1, b
    return best_b, best_f1

In [ ]:
def build_param_groups(_model):
    decay_summ, decay_other, no_decay = [], [], []
    for n, p in _model.named_parameters():
        if not p.requires_grad:
            continue

        # Biases & (LayerNorm/BatchNorm/etc.) weights are 1-D -> no weight decay
        is_no_decay = (p.dim() == 1) or n.endswith('.bias') or 'norm' in n.lower() or 'bn' in n.lower()

        if is_no_decay:
            no_decay.append(p)
        elif n.startswith('summ_mlp'):
            decay_summ.append(p)
        else:
            decay_other.append(p)

    # Optional sanity checks to catch overlaps or misses
    ids = [id(x) for x in decay_summ + decay_other + no_decay]
    assert len(ids) == len(set(ids)), "A parameter landed in more than one group."
    total = sum(p.numel() for p in _model.parameters() if p.requires_grad)
    covered = sum(p.numel() for p in decay_summ + decay_other + no_decay)
    assert total == covered, f"Some params not assigned: total {total} vs covered {covered}"

    return [
        {'params': decay_summ,  'weight_decay': 5e-4},
        {'params': decay_other, 'weight_decay': 1e-4},
        {'params': no_decay,    'weight_decay': 0.0},
    ]

def set_requires_grad(module, flag: bool):
    """
    Set requires_grad for all parameters in a module.
    :param module:
    :param flag:
    :return:
    """
    for p in module.parameters():
        p.requires_grad = flag

def train_one_fold(
        _k, _train_loader, _val_loader, y_tr, _class2_multiplier,
        epochs=160, patience=24, warmup_epochs=5, p_drop_summ=0.3
):
    # ----- per-fold class weights (with class-2 boost) -----
    base_w = compute_class_weight('balanced', classes=np.array([0,1,2]), y=y_tr).astype(np.float32)
    base_w[2] *= _class2_multiplier
    class_weights = torch.tensor(base_w, dtype=torch.float32, device=device)

    _model = PainTCNBiLSTMAttn(
        d_num=90,
        d_emb=2,
        d_summ=data.X_dyn_summ_train.shape[1],
        tcn_channels=128,
        lstm_hidden=128,
        num_classes=3,
        dropout=0.3
    ).to(device)

    # --- WARM-UP: freeze summary branch and/or hard-mask it ---
    set_requires_grad(model.summ_mlp, False)
    model.p_drop_summ = 1.0   # always drop summary during warm-up

     # build optimizer on current trainable params
    criterion = torch.nn.CrossEntropyLoss(weight=class_weights)
    optimizer = torch.optim.AdamW(build_param_groups(_model), lr=1e-3)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode='max',
        factor=0.5,
        patience=5,
        cooldown=3, # after lr reduction, wait this many epochs before resuming normal operation
        min_lr=2e-5
    )


    # ---- warm-up loop ----
    best_f1, best_state, bad = -1.0, None, 0
    for epoch in range(1, warmup_epochs+1):
        # TODO: generalize, duplicated!
        _model.train()
        running = 0.0
        for batch in _train_loader:
            x_num  = batch['x_num'].to(device)
            x_sta  = batch['x_sta'].to(device)
            x_surv = [s.to(device) for s in batch['x_surv']]
            x_summ = batch['x_summ'].to(device)
            y      = batch['y'].to(device)

            optimizer.zero_grad()
            logits = _model(x_num, x_surv, x_sta, x_summ)
            loss   = criterion(logits, y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(_model.parameters(), 1.0)
            optimizer.step()
            running += loss.item() * y.size(0)

        train_loss = running / len(_train_loader.dataset)

        # --- validation ---
        macro_f1, rec_pc, cm = evaluate_metrics(_model, _val_loader, device)
        scheduler.step(macro_f1)

        # where lr means current learning rate
        # and rec is per-class recall
        print(f"[F{_k} {epoch:03d}] t_loss={train_loss:.4f} | F1(macro)={macro_f1:.4f} | "
              f"rec={np.round(rec_pc,3)} | lr={optimizer.param_groups[0]['lr']:.2e} | "
              f"patience={bad+1}/{patience}")

        # early stopping on macro-F1
        if macro_f1 > best_f1:
            best_f1, bad = macro_f1, 0
            best_state = {kk: vv.cpu() for kk, vv in _model.state_dict().items()}
        else:
            bad += 1
            if bad >= patience:
                print(f"[F{_k}] Early stopping.")
                break

    # --- UNFREEZE summaries & set chosen drop prob (e.g., 0.3 or 0.7) ---
    set_requires_grad(model.summ_mlp, True)
    model.p_drop_summ = p_drop_summ

    # re-build optimizer so the newly-unfrozen params get included with correct WD
    param_groups = build_param_groups(model)
    optimizer = torch.optim.AdamW(param_groups, lr=1e-3)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max',
                                                           factor=0.5, patience=5, cooldown=3, min_lr=2e-5)

    # ---- main training loop ----
    best_f1, best_state, bad = -1.0, None, 0
    for epoch in range(1, epochs+1):
        _model.train()
        running = 0.0
        for batch in _train_loader:
            x_num  = batch['x_num'].to(device)
            x_sta  = batch['x_sta'].to(device)
            x_surv = [s.to(device) for s in batch['x_surv']]
            x_summ = batch['x_summ'].to(device)
            y      = batch['y'].to(device)

            optimizer.zero_grad()
            logits = _model(x_num, x_surv, x_sta, x_summ)
            loss   = criterion(logits, y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(_model.parameters(), 1.0)
            optimizer.step()
            running += loss.item() * y.size(0)

        train_loss = running / len(_train_loader.dataset)

        # --- validation ---
        macro_f1, rec_pc, cm = evaluate_metrics(_model, _val_loader, device)
        scheduler.step(macro_f1)

        # where lr means current learning rate
        # and rec is per-class recall
        print(f"[F{_k} {epoch:03d}] t_loss={train_loss:.4f} | F1(macro)={macro_f1:.4f} | "
              f"rec={np.round(rec_pc,3)} | lr={optimizer.param_groups[0]['lr']:.2e} | "
              f"patience={bad+1}/{patience}")

        # early stopping on macro-F1
        if macro_f1 > best_f1:
            best_f1, bad = macro_f1, 0
            best_state = {kk: vv.cpu() for kk, vv in _model.state_dict().items()}
        else:
            bad += 1
            if bad >= patience:
                print(f"[F{_k}] Early stopping.")
                break

    # restore best
    _model.load_state_dict({kk: vv.to(device) for kk, vv in best_state.items()})
    _best_b, f1_biased = search_logit_bias(model, val_loader, yva, _device=device)
    print("Best logit bias:", _best_b, "biased macro-F1:", f1_biased)
    final_f1, final_rec, final_cm = evaluate_metrics(_model, _val_loader, device)
    torch.save(best_state, f'artifacts/fold{_k}_best_K_{_class2_multiplier}mul.pt')
    try:
        json.dump({'macroF':final_f1, 'recall':final_rec.tolist(), 'cm':final_cm.tolist(), 'window': data.feat_eng['window']},
                  open(f'artifacts/fold{_k}_metrics_{_class2_multiplier}mul.json', 'w'), indent=4)
    except Exception as e:
        print("Could not save metrics json:", e)
    print(f"[F{_k}] BEST macroF={final_f1:.4f}  saved=artifacts/fold{_k}_best_K_{_class2_multiplier}mul.pt")
    return _model, final_f1, _best_b

In [ ]:
# --- build 5 folds and run ---
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_scores = []
test_fold_logits = []
y_all = np.concatenate([data.y_train, data.y_val], axis=0)
X_num_all = np.concatenate([data.X_dyn_num_train, data.X_dyn_num_val], axis=0) # (N,160,90)
X_sta_all = np.concatenate([data.X_sta_train, data.X_sta_val], axis=0)       # (N,7)
X_surv_all = [np.concatenate([data.X_surv_train[i], data.X_surv_val[i]], axis=0) for i in range(4)]
X_summ_all = np.concatenate([data.X_dyn_summ_train, data.X_dyn_summ_val], axis=0) # (N,F_summ)
class2_multipliers = [
    1, 1.05, 1.1, 1.15, 1.2, 1.25, 1.3, 1.35, 1.4, 1.45, 1.5
]

tg.send_message("🚀 Starting K-Fold training for PainTCNBiLSTMAttn model...", False)
for k, (tr_idx, va_idx) in enumerate(skf.split(np.zeros(len(y_all)), y_all)):
    for class2_multiplier in class2_multipliers:
        print(f"=== Fold {k} with class-2 multiplier {class2_multiplier} ===")
        Xtr_num, Xva_num = X_num_all[tr_idx], X_num_all[va_idx]
        Xtr_sta, Xva_sta = X_sta_all[tr_idx], X_sta_all[va_idx]
        Xtr_surv = [s[tr_idx] for s in X_surv_all]
        Xva_surv = [s[va_idx] for s in X_surv_all]
        Xtr_summ, Xva_summ = X_summ_all[tr_idx], X_summ_all[va_idx]
        ytr, yva = y_all[tr_idx], y_all[va_idx]

        train_ds = PainDataset(Xtr_num, Xtr_surv, Xtr_sta, Xtr_summ, ytr)
        val_ds   = PainDataset(Xva_num, Xva_surv, Xva_sta, Xva_summ, yva)

        train_loader = DataLoader(train_ds, batch_size=64, shuffle=True,  num_workers=4, pin_memory=True)
        # do not shuffle validation since we want consistent metrics across epochs
        val_loader   = DataLoader(val_ds,   batch_size=64, shuffle=False, num_workers=4, pin_memory=True)

        model, f1, best_b = train_one_fold(k, train_loader, val_loader, ytr, _class2_multiplier=class2_multiplier, epochs=1000, patience=24)
        fold_scores.append(f1)

        # ---- (optional) test logits for ensembling ----
        logits_te = infer_logits(model, data.X_dyn_num_test, data.X_surv_test, data.X_sta_test, data.X_dyn_summ_test, batch_size=64)  # (1324,3)
        logits_te = logits_te + best_b[None, :]
        pred_idx  = logits_te.argmax(1)
        test_fold_logits.append(logits_te)

        # --- GPU / CPU stats after each fold ---
        try:
            for gpu in GPUtil.getGPUs():
                print(f"{gpu.name} ({gpu.id}) Usage after fold {k} class2_mul {class2_multiplier}:")
                print(f"    Temp: {gpu.temperature}C, Load: {gpu.load*100:.1f}%, Mem Load: {gpu.memoryUtil*100:.1f}%")
        except Exception as e:
            print("Could not get GPU stats:", e)
        try:
            cores = psutil.sensors_temperatures().get('coretemp', [])
            if cores:
                for core in cores:
                    print(f"CPU Core {core.label} Temp: {core.current}C")
        except Exception as e:
            print("Could not get CPU temperature:", e)
        sleep(3) # cooldown between folds

print("CV macro-F1 mean:", float(np.mean(fold_scores)), "std:", float(np.std(fold_scores)))
# send a telegram chat notification to notify completion
tg.send_message(f"✅ Fold training complete. CV macro-F1 mean: {float(np.mean(fold_scores)):.4f} std: {float(np.std(fold_scores)):.4f}", False)


In [ ]:
# --- ensemble test logits and write submission ---
if len(test_fold_logits) > 0:
    logits_stack = np.stack(test_fold_logits, axis=0)   # (5,1324,3)
    logits_avg   = logits_stack.mean(axis=0)            # (1324,3)
    pred_idx     = logits_avg.argmax(axis=1)            # (1324,)
    idx_to_label = {0:'no_pain', 1:'low_pain', 2:'high_pain'}
    pred_labels  = [idx_to_label[int(i)] for i in pred_idx]

    ids_test = [f'{i:03}' for i in range(1324)]

    sub = pd.DataFrame({'sample_index': ids_test, 'label': pred_labels})
    sub.to_csv('submissions/submission_cv_tcn_bilstm_attn_w5_m2-1p3.csv', index=False)
    print("Saved:", 'submissions/submission_cv_tcn_bilstm_attn_w5_m2-1p3.csv')
    try:
        tg.send_file(
            'submissions/submission_cv_tcn_bilstm_attn_w5_m2-1p3.csv',
            caption="✅ Submission file generated from CV ensemble."
        )
    except Exception as e:
        print("Could not send telegram file:", e)
